# Traffic Demand Prediction System (Key-Lookup + Fallback)
This notebook implements a spatiotemporal key-lookup strategy to predict traffic demand, with a tabular regression fallback for missing historical matches.

## Step 1: Imports and Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import OrdinalEncoder
import warnings
warnings.filterwarnings('ignore')

## Step 2: Preprocessing and Utility Functions

In [ ]:
import pandas as pd
import numpy as np

def decode_geohash(geohash):
    """
    Decodes a geohash string into latitude and longitude.
    """
    if not isinstance(geohash, str) or len(geohash) == 0:
        return np.nan, np.nan
    base32 = '0123456789bcdefghjkmnpqrstuvwxyz'
    lat_interval = (-90.0, 90.0)
    lon_interval = (-180.0, 180.0)
    is_even = True
    
    for char in geohash:
        if char not in base32:
            return np.nan, np.nan
        val = base32.index(char)
        for i in range(4, -1, -1):
            bit = (val >> i) & 1
            if is_even:
                mid = (lon_interval[0] + lon_interval[1]) / 2.0
                if bit == 1:
                    lon_interval = (mid, lon_interval[1])
                else:
                    lon_interval = (lon_interval[0], mid)
            else:
                mid = (lat_interval[0] + lat_interval[1]) / 2.0
                if bit == 1:
                    lat_interval = (mid, lat_interval[1])
                else:
                    lat_interval = (lat_interval[0], mid)
            is_even = not is_even
            
    lat = (lat_interval[0] + lat_interval[1]) / 2.0
    lon = (lon_interval[0] + lon_interval[1]) / 2.0
    return lat, lon

def parse_time_features(df):
    """
    Extracts time features from the timestamp column.
    timestamp format is 'H:M' (e.g. '2:15')
    """
    df = df.copy()
    df['hour'] = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df['minute'] = df['timestamp'].apply(lambda x: int(x.split(':')[1]))
    df['time_minutes'] = df['hour'] * 60 + df['minute']
    
    # Cyclic features
    df['sin_time'] = np.sin(2 * np.pi * df['time_minutes'] / 1440.0)
    df['cos_time'] = np.cos(2 * np.pi * df['time_minutes'] / 1440.0)
    
    return df

def impute_missing_values(df, train_df=None):
    """
    Imputes missing values for RoadType, Weather, and Temperature.
    If train_df is provided, uses it to compute imputation mappings.
    """
    df = df.copy()
    ref_df = train_df if train_df is not None else df
    
    # Modes
    road_mode = ref_df['RoadType'].mode()[0] if not ref_df['RoadType'].mode().empty else 'Residential'
    weather_mode = ref_df['Weather'].mode()[0] if not ref_df['Weather'].mode().empty else 'Sunny'
    
    df['RoadType'] = df['RoadType'].fillna(road_mode)
    df['Weather'] = df['Weather'].fillna(weather_mode)
    
    # Temperature mapping: Group by hour and Weather to get mean temperature
    temp_map = ref_df.groupby(['hour', 'Weather'])['Temperature'].mean().to_dict()
    overall_mean = ref_df['Temperature'].mean()
    if np.isnan(overall_mean):
        overall_mean = 20.0
        
    def fill_temp(row):
        if not np.isnan(row['Temperature']):
            return row['Temperature']
        key = (row['hour'], row['Weather'])
        if key in temp_map and not np.isnan(temp_map[key]):
            return temp_map[key]
        return overall_mean
        
    df['Temperature'] = df.apply(fill_temp, axis=1)
    return df


## Step 3: Feature Engineering Pipeline

In [ ]:
def build_features(df, train_df=None, is_train=True):
    """
    Builds purely raw environmental and geospatial features for df.
    Eliminates target-leaky lag features.
    """
    # 1. Parse basic time features
    df = parse_time_features(df.copy())
    
    # 2. Decode geohashes
    lats_lons = df['geohash'].apply(decode_geohash)
    df['latitude'] = [x[0] for x in lats_lons]
    df['longitude'] = [x[1] for x in lats_lons]
    
    # 3. Geohash prefixes
    df['geohash_prefix4'] = df['geohash'].str[:4]
    df['geohash_prefix5'] = df['geohash'].str[:5]
    
    # 4. Impute missing values
    if train_df is not None:
        train_parsed = parse_time_features(train_df.copy())
        df = impute_missing_values(df, train_parsed)
    else:
        df = impute_missing_values(df)
        
    # Final NaNs safety check for features only (not target)
    feature_cols = [c for c in df.columns if c != 'demand']
    df[feature_cols] = df[feature_cols].fillna(0.0)
    
    return df


## Step 4: Loading Data and Setting Up Spatiotemporal Key-Lookup

In [ ]:
print("Loading Datasets...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print("Preprocessing and Feature Engineering...")
# Concatenate train and test to ensure consistent feature engineering
all_df = pd.concat([train, test], ignore_index=True)
all_feat = build_features(all_df)

# Split back to train and test
train_feat = all_feat[all_feat['demand'].notna()].copy()
test_feat = all_feat[all_feat['demand'].isna()].copy()

# Categorical encoding
cat_cols = ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks', 'geohash', 'geohash_prefix4', 'geohash_prefix5']
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# Fit encoder on train and transform both
train_feat[cat_cols] = encoder.fit_transform(train_feat[cat_cols].astype(str))
test_feat[cat_cols] = encoder.transform(test_feat[cat_cols].astype(str))

features = [
    'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather',
    'hour', 'minute', 'time_minutes', 'sin_time', 'cos_time', 'latitude', 'longitude',
    'geohash_prefix4', 'geohash_prefix5'
]

target = 'demand'

# Split into Day 48 and Day 49
train_48 = train_feat[train_feat['day'] == 48].copy()

print(f"Train Day 48 shape: {train_48.shape}")
print(f"Test Day 49 shape: {test_feat.shape}")

# --- SPATIOTEMPORAL KEY-LOOKUP SETUP ---
# Create the lookup dictionary mapping (geohash, time_minutes) to demand from Day 48
train_48_original = train[train['day'] == 48].copy()
train_48_original['time_minutes'] = train_48_original['timestamp'].apply(lambda x: int(x.split(':')[0])*60 + int(x.split(':')[1]))
lookup_dict = train_48_original.set_index(['geohash', 'time_minutes'])['demand'].to_dict()

test_original = test.copy()
test_original['time_minutes'] = test_original['timestamp'].apply(lambda x: int(x.split(':')[0])*60 + int(x.split(':')[1]))


## Step 5: Training Fallback Model (Pure Tabular Regression)

In [ ]:
# K-Fold Cross Validation Setup on Day 48 for the fallback model
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Test prediction arrays
test_lgb = np.zeros(len(test_feat))
test_xgb = np.zeros(len(test_feat))
test_cb = np.zeros(len(test_feat))

lgb_params = {
    'n_estimators': 1500,
    'learning_rate': 0.03,
    'num_leaves': 63,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbose': -1
}

xgb_params = {
    'n_estimators': 1500,
    'learning_rate': 0.03,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbosity': 0
}

cb_params = {
    'iterations': 1500,
    'learning_rate': 0.03,
    'depth': 6,
    'random_seed': 42,
    'verbose': 0
}

for fold, (train_idx, val_idx) in enumerate(kf.split(train_48)):
    print(f"\n--- Training Fold {fold} ---")
    
    X_train, y_train = train_48.iloc[train_idx][features], train_48.iloc[train_idx][target]
    X_val, y_val = train_48.iloc[val_idx][features], train_48.iloc[val_idx][target]
    
    # 1. LightGBM
    model_lgb = lgb.LGBMRegressor(**lgb_params)
    model_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
    test_lgb += np.clip(model_lgb.predict(test_feat[features]), 0.0, 1.0) / 5.0
    
    # 2. XGBoost
    model_xgb = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=50)
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    test_xgb += np.clip(model_xgb.predict(test_feat[features]), 0.0, 1.0) / 5.0
    
    # 3. CatBoost
    model_cb = cb.CatBoostRegressor(**cb_params, early_stopping_rounds=50)
    model_cb.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)
    test_cb += np.clip(model_cb.predict(test_feat[features]), 0.0, 1.0) / 5.0


## Step 6: Generate Submission using Lookup + Fallback

In [ ]:
fallback_preds = (test_lgb + test_xgb + test_cb) / 3.0

final_predictions = []
lookup_count = 0
fallback_count = 0

for idx, row in test_original.iterrows():
    key = (row['geohash'], row['time_minutes'])
    if key in lookup_dict:
        final_predictions.append(lookup_dict[key])
        lookup_count += 1
    else:
        final_predictions.append(fallback_preds[idx])
        fallback_count += 1
        
print(f"Predictions from exact historical matches: {lookup_count}")
print(f"Predictions from regression fallback model: {fallback_count}")

# Construct submission dataframe
submission = pd.DataFrame({
    'Index': test_feat['Index'].astype(int),
    'demand': np.clip(final_predictions, 0.0, 1.0)
})

submission.to_csv('submission.csv', index=False)
print("Submission successfully saved and verified!")
